# 01 — Acquisition des données

**Objectif du projet :** prédire le classement des JO de Los Angeles 2028 (pays + médailles par sport et par type).

Trois sources suffisent. Elles sont versionnées directement dans le dépôt (`data/`) — aucun téléchargement requis.

| Fichier | Rôle | Lignes |
|---|---|---|
| `data/JO_resultats.csv` | Résultats olympiques 1896→2024 (1 ligne = 1 athlète médaillé) | ~24 000 |
| `data/JO2028_sports.csv` | Programme officiel des disciplines de LA 2028 | ~350 |
| `data/JO_date_ete.csv` | Table de référence des éditions d'été (année, ville hôte, pays hôte) 1896→2032 | ~32 |

Ce notebook se contente de **charger et vérifier** ces trois fichiers. Le nettoyage et
l'agrégation sont faits dans `02_preparation.ipynb`.

## Imports et chemins

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

PROJECT_ROOT = Path.cwd().parent if (Path.cwd().name == "notebooks") else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

RESULTS_CSV = DATA_DIR / "JO_resultats.csv"
SPORTS_2028_CSV = DATA_DIR / "JO2028_sports.csv"
DATES_CSV = DATA_DIR / "JO_date_ete.csv"

assert RESULTS_CSV.exists(), f"Fichier manquant : {RESULTS_CSV}"
assert SPORTS_2028_CSV.exists(), f"Fichier manquant : {SPORTS_2028_CSV}"
assert DATES_CSV.exists(), f"Fichier manquant : {DATES_CSV}"
print("Sources presentes dans le depot :")
print(f"  - {RESULTS_CSV.name}")
print(f"  - {SPORTS_2028_CSV.name}")
print(f"  - {DATES_CSV.name}")

Sources presentes dans le depot :
  - JO_resultats.csv
  - JO2028_sports.csv
  - JO_date_ete.csv


## 1. `JO_resultats.csv` — Résultats olympiques 1896-2024

Source historique des médailles. Une ligne par athlète médaillé : pour les épreuves
collectives (relais, sports d'équipe), chaque membre apparaît sur une ligne distincte.
Le séparateur est `;`.

In [2]:
resultats = pd.read_csv(RESULTS_CSV, sep=";", low_memory=False)
print(f"Dimensions : {resultats.shape[0]:,} lignes x {resultats.shape[1]} colonnes")
print(f"\nColonnes :")
print(resultats.dtypes)
resultats.head()

Dimensions : 24,012 lignes x 10 colonnes

Colonnes :
discipline        str
event             str
event_gender      str
event_type        str
medal_type        str
name              str
country_code      str
country           str
city              str
event_year      int64
dtype: object


,discipline,event,event_gender,event_type,medal_type,name,country_code,country,city,event_year
0,Cycling Road,Men Individual Time Trial,Male,Athlete,Gold,Evenepoel Remco,BEL,Belgium,Paris,2024
1,Cycling Road,Men Individual Time Trial,Male,Athlete,Silver,Ganna Filippo,ITA,Italy,Paris,2024
2,Cycling Road,Men Individual Time Trial,Male,Athlete,Bronze,Van Aert Wout,BEL,Belgium,Paris,2024
3,Cycling Road,Women Individual Time Trial,Female,Athlete,Gold,Brown Grace,AUS,Australia,Paris,2024
4,Cycling Road,Women Individual Time Trial,Female,Athlete,Silver,Henderson Anna,GBR,Great Britain,Paris,2024


In [3]:
# Couverture et distribution
print(f"Periode          : {resultats['event_year'].min()} -> {resultats['event_year'].max()}")
print(f"Editions         : {resultats['event_year'].nunique()}")
print(f"Pays (NOC)       : {resultats['country_code'].nunique()}")
print(f"Disciplines      : {resultats['discipline'].nunique()}")
print(f"\nRepartition par type de medaille :")
print(resultats['medal_type'].value_counts())

Periode          : 1896 -> 2024
Editions         : 38
Pays (NOC)       : 160
Disciplines      : 87

Repartition par type de medaille :
medal_type
Bronze    8336
Gold      7861
Silver    7815
Name: count, dtype: int64


In [4]:
# Valeurs manquantes
print("Valeurs manquantes par colonne :")
print(resultats.isnull().sum())

Valeurs manquantes par colonne :
discipline         0
event              0
event_gender       0
event_type         0
medal_type         0
name            3624
country_code       0
country            0
city               0
event_year         0
dtype: int64


**Observations sur `JO_resultats.csv` :**
- Couvre l'intégralité de la période moderne (1896-2024), Paris 2024 inclus.
- Les médailles collectives sont dupliquées par athlète → une **déduplication** sera
  nécessaire dans `02_preparation` pour ne compter qu'une médaille par épreuve.
- Les colonnes clés sont propres (`medal_type`, `country_code`, `event_year`).

## 2. `JO2028_sports.csv` — Programme officiel LA 2028

Liste des disciplines et épreuves au programme des Jeux de Los Angeles 2028.
Permet de connaître précisément le périmètre à prédire et d'identifier les
**sports nouveaux** (sans historique, donc non modélisables).

In [5]:
sports_2028 = pd.read_csv(SPORTS_2028_CSV, sep=";")
print(f"Dimensions : {sports_2028.shape[0]} epreuves x {sports_2028.shape[1]} colonnes")
print(f"Disciplines distinctes : {sports_2028['Discipline'].nunique()}")
print(f"\nColonnes : {sports_2028.columns.tolist()}")
sports_2028.head(10)

Dimensions : 352 epreuves x 3 colonnes
Disciplines distinctes : 51

Colonnes : ['Discipline', 'event', 'event_gender']


,Discipline,event,event_gender
0,3x3 Basketball,Men,Male
1,3x3 Basketball,Women,Female
2,Archery,Men Individual,Male
3,Archery,Women Individual,Female
4,Archery,Mixed Team,Mixed
5,Archery,Men Team,Male
6,Archery,Women Team,Female
7,Archery,Compound Mixed Team,Mixed
8,Artistic Gymnastics,Men Team,Male
9,Artistic Gymnastics,Men All Around,Male


In [6]:
# Repartition des epreuves par genre
print("Repartition des epreuves par genre :")
print(sports_2028['event_gender'].value_counts())

Repartition des epreuves par genre :
event_gender
Male         165
Female       162
Mixed         19
Undefined      6
Name: count, dtype: int64


**Observations sur `JO2028_sports.csv` :**
- Donne le périmètre exact de LA 2028 (disciplines + épreuves).
- Servira dans `02_preparation` à repérer par différence ensembliste les sports
  jamais disputés auparavant (cricket, flag football, squash...), que le modèle ne
  pourra pas prédire faute d'historique.

## 3. `JO_date_ete.csv` — Table de référence des éditions d'été

Une ligne par édition **d'été** : année, ville hôte, pays hôte, de 1896 à 2032.
`JO_resultats.csv` mélange été et hiver (jusqu'en 1992 ils tombaient la même année) ;
ce fichier sert de clé pour isoler proprement l'été dans `02_preparation` par jointure
sur le couple (année, ville), sans aucune liste écrite en dur.

In [7]:
dates_ete = pd.read_csv(DATES_CSV, sep=";")
print(f"Dimensions : {dates_ete.shape[0]} editions x {dates_ete.shape[1]} colonnes")
print(f"Colonnes   : {dates_ete.columns.tolist()}")
print(f"Periode    : {dates_ete['event_year'].min()} -> {dates_ete['event_year'].max()}")

# Editions deja disputees (servent a l'historique) vs futures (2028, 2032)
disputees = dates_ete[dates_ete['event_year'] <= 2024]
futures = dates_ete[dates_ete['event_year'] > 2024]
print(f"\nEditions disputees (<=2024) : {len(disputees)}")
print(f"Editions futures (>2024)    : {len(futures)} -> {futures['event_year'].tolist()}")
dates_ete.head(10)

Dimensions : 32 editions x 3 colonnes
Colonnes   : ['event_year', 'city', 'country']
Periode    : 1896 -> 2032

Editions disputees (<=2024) : 30
Editions futures (>2024)    : 2 -> [2028, 2032]


,event_year,city,country
0,1896,Athens,Greece
1,1900,Paris,France
2,1904,St. Louis,United States
3,1908,London,United Kingdom
4,1912,Stockholm,Sweden
5,1920,Antwerp,Belgium
6,1924,Paris,France
7,1928,Amsterdam,Netherlands
8,1932,Los Angeles,United States
9,1936,Berlin,Germany


## Synthèse

Les deux sources sont chargées et cohérentes. La suite du pipeline :
1. `02_preparation.ipynb` — nettoyage, déduplication, agrégation `(pays × année × sport)`,
   validation vs chiffres officiels CIO.
2. `03_modelisation.ipynb` — 3 modèles Ridge (Or, Argent, Bronze) + projection LA 2028.

In [8]:
print("=== Recapitulatif acquisition ===")
print(f"JO_resultats.csv   : {len(resultats):,} lignes, {resultats['event_year'].nunique()} editions")
print(f"JO2028_sports.csv  : {len(sports_2028)} epreuves, {sports_2028['Discipline'].nunique()} disciplines")
print(f"JO_date_ete.csv    : {len(dates_ete)} editions d'ete ({dates_ete['event_year'].min()}->{dates_ete['event_year'].max()})")
print("\nDonnees pretes pour 02_preparation.ipynb")

=== Recapitulatif acquisition ===
JO_resultats.csv   : 24,012 lignes, 38 editions
JO2028_sports.csv  : 352 epreuves, 51 disciplines
JO_date_ete.csv    : 32 editions d'ete (1896->2032)

Donnees pretes pour 02_preparation.ipynb
